1. Load, Impute, Scale, and Split Data

We load the cleaned data, drop unnecessary columns, convert categorical features, and apply imputation and scaling prior to splitting to avoid data leakage and missing value errors during model training.

In [18]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Load cleaned data
df = pd.read_csv("/Users/rasheedmehrinfar/Desktop/springboard/capstone-two/Data/cleaned_co2_data.csv")

# Drop identifier/redundant columns
df_model = df.copy()
target = 'co2'
cols_to_drop = ['iso_code', 'co2_change']
df_model = df_model.drop(columns=cols_to_drop)

# One-hot encode 'country'
df_model = pd.get_dummies(df_model, columns=['country'], drop_first=True)

# Drop rows where target is NaN
df_model = df_model[df_model[target].notna()]

# Separate features and target
X = df_model.drop(columns=[target])
y = df_model[target]

# Impute missing values BEFORE splitting
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

2. Linear Regression

Linear Regression is used as a simple and interpretable baseline model.

In [21]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression:")
print(f"MAE: {mae_lr:.2f}, RMSE: {rmse_lr:.2f}, R2: {r2_lr:.2f}")

Linear Regression:
MAE: 0.31, RMSE: 0.83, R2: 1.00



3. Random Forest Regressor

Random Forest is a powerful tree-based ensemble model. We apply grid search to tune its key hyperparameters.

In [24]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

rfr = RandomForestRegressor(random_state=42)
param_grid_rf = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10]
}
grid_rf = GridSearchCV(rfr, param_grid_rf, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_rf.fit(X_train, y_train)
y_pred_rf = grid_rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Regressor:")
print(f"Best Params: {grid_rf.best_params_}")
print(f"MAE: {mae_rf:.2f}, RMSE: {rmse_rf:.2f}, R2: {r2_rf:.2f}")

Random Forest Regressor:
Best Params: {'max_depth': None, 'n_estimators': 50}
MAE: 3.73, RMSE: 28.55, R2: 1.00


4. XGBoost Regressor

XGBoost is a high-performance boosting model known for handling complex data patterns.

In [27]:
from xgboost import XGBRegressor

xgb = XGBRegressor(random_state=42)
param_grid_xgb = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5]
}
grid_xgb = GridSearchCV(xgb, param_grid_xgb, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_xgb.fit(X_train, y_train)
y_pred_xgb = grid_xgb.predict(X_test)

mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print("XGBoost Regressor:")
print(f"Best Params: {grid_xgb.best_params_}")
print(f"MAE: {mae_xgb:.2f}, RMSE: {rmse_xgb:.2f}, R2: {r2_xgb:.2f}")

XGBoost Regressor:
Best Params: {'max_depth': 3, 'n_estimators': 100}
MAE: 7.32, RMSE: 42.07, R2: 1.00


5. Compare Model Performance

Finally, we compare all trained models using MAE, RMSE, and R² Score to decide which model performed best.

In [30]:
import pandas as pd

results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost'],
    'MAE': [mae_lr, mae_rf, mae_xgb],
    'RMSE': [rmse_lr, rmse_rf, rmse_xgb],
    'R2 Score': [r2_lr, r2_rf, r2_xgb]
})

print("Model Comparison:")
print(results.sort_values(by='RMSE'))

Model Comparison:
               Model       MAE       RMSE  R2 Score
0  Linear Regression  0.310184   0.829026  0.999999
1      Random Forest  3.725606  28.550122  0.998920
2            XGBoost  7.319794  42.072347  0.997654


We can see Linear Regression seems to be the best model in this case. Despite being the simplest model:

- It predicts extremely close to the actual values (very low MAE/RMSE).

- It explains nearly 100% of the variance in the target (R² ≈ 1).

- It also has the lowest computational cost, making it scalable and interpretable.
